# Relatório operacional de pré-decolagem

Notebook didático da issue #4. Ele executa cenários determinísticos do repositório e integra leitura, validação, energia e decisão. As faixas usadas são hipóteses didáticas; não representam parâmetros certificados de uma nave real.

## Onde estão os requisitos da atividade

| Requisito | Evidência no projeto |
| --- | --- |
| 1.1 Telemetria | Tabelas e unidades em [telemetria](../docs/telemetria.md); leituras nos cenários abaixo. |
| 1.2 Algoritmo | [Regras e percursos](../docs/algoritmo.md), [pseudocódigo](../src/pseudocodigo_verificacao.md). |
| 1.3 Python | Leitura, validação, verificação e impressão nas células deste notebook. |
| 1.4 Energia | Capacidade, carga, consumo, perdas e autonomia em [energia](../docs/energia.md) e resultados abaixo. |
| 1.5 Análise por IA | [Prompt, resposta, classificação, anomalias, riscos e revisão](../docs/analise-ia.md). |
| 1.6 Reflexão | [Ética, impacto social e sustentabilidade](../docs/reflexao_critica.md). |

**Atenção:** execute primeiro a célula Preparação ou use Run All. O gerador opcional no fim sorteia dados novos. A análise assistida por IA do item 1.5 é uma consulta documentada; o gerador não substitui essa análise.


## Preparação

O notebook funciona quando aberto na raiz do repositório ou dentro da pasta `notebooks/`. Não requer API key nem acesso à internet.

In [1]:
from pathlib import Path
import json
import sys

CANDIDATOS = (Path.cwd(), Path.cwd().parent)
RAIZ = next(p for p in CANDIDATOS if (p / 'src').is_dir())
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.apresentacao import formatar_resultado
from src.missao import LIMITES_PADRAO, executar_cenario


def exibir_telemetria(nome_arquivo):
    """Mostra os valores de entrada com as respectivas unidades."""
    dados = json.loads((RAIZ / 'dados' / nome_arquivo).read_text(encoding='utf-8'))
    print(f'Entrada: {nome_arquivo}')
    print(f"Temperatura interna: {dados['temperatura_interna_c']} °C")
    print(f"Temperatura externa: {dados['temperatura_externa_c']} °C")
    print(f"Energia: {dados['energia_pct']}%")
    print(f"Pressão do tanque: {dados['pressao_tanque_kpa']} kPa")
    print(f"Integridade estrutural: {dados['integridade_estrutural']}")
    modulos = ', '.join(f'{nome}={estado}' for nome, estado in dados['modulos'].items())
    print(f'Módulos críticos: {modulos}')

print('Repositório: raiz do projeto')
print(f'Limites didáticos: {LIMITES_PADRAO}')

Repositório: raiz do projeto
Limites didáticos: {'temperatura_interna_c': (15, 30), 'temperatura_externa_c': (-150, 120), 'energia_pct': (50, 100), 'pressao_tanque_kpa': (90, 110)}


## Apresentação dos resultados

A orquestração devolve dados estruturados e `formatar_resultado` os converte em texto legível, sem recalcular energia ou alterar a decisão.

## Cenário nominal

O cenário nominal deve liberar a decolagem.

In [2]:
exibir_telemetria('nominal.json')
nominal = executar_cenario(RAIZ / 'dados' / 'nominal.json', LIMITES_PADRAO)
print(formatar_resultado(nominal))

Entrada: nominal.json
Temperatura interna: 22 °C
Temperatura externa: -50 °C
Energia: 80%
Pressão do tanque: 100 kPa
Integridade estrutural: 1
Módulos críticos: suporte_vida=OK, energia=OK, comunicacao=OK, propulsao=OK, navegacao=OK
Cenário: nominal.json
Decisão: PRONTO PARA DECOLAR
Nenhuma falha operacional identificada.
Energia inicial: 80.00 kWh
Perdas: 4.00 kWh
Energia útil: 76.00 kWh
Saldo após decolagem: 56.00 kWh
Autonomia estimada: 5.60 h


## Falha de temperatura

A leitura interna acima da faixa segura é válida como dado, mas a decisão deve abortar e informar o motivo.

In [3]:
exibir_telemetria('falha_temperatura.json')
falha_temperatura = executar_cenario(RAIZ / 'dados' / 'falha_temperatura.json', LIMITES_PADRAO)
print(formatar_resultado(falha_temperatura))

Entrada: falha_temperatura.json
Temperatura interna: 31 °C
Temperatura externa: -50 °C
Energia: 80%
Pressão do tanque: 100 kPa
Integridade estrutural: 1
Módulos críticos: suporte_vida=OK, energia=OK, comunicacao=OK, propulsao=OK, navegacao=OK
Cenário: falha_temperatura.json
Decisão: DECOLAGEM ABORTADA
Motivos:
- temperatura_interna_c 31 acima do maximo de 30
Energia inicial: 80.00 kWh
Perdas: 4.00 kWh
Energia útil: 76.00 kWh
Saldo após decolagem: 56.00 kWh
Autonomia estimada: 5.60 h


## Falha energética

A telemetria é válida, porém o saldo depois de perdas e consumo é insuficiente.

In [4]:
exibir_telemetria('falha_energia.json')
falha_energia = executar_cenario(RAIZ / 'dados' / 'falha_energia.json', LIMITES_PADRAO)
print(formatar_resultado(falha_energia))

Entrada: falha_energia.json
Temperatura interna: 22 °C
Temperatura externa: -50 °C
Energia: 80%
Pressão do tanque: 100 kPa
Integridade estrutural: 1
Módulos críticos: suporte_vida=OK, energia=OK, comunicacao=OK, propulsao=OK, navegacao=OK
Cenário: falha_energia.json
Decisão: DECOLAGEM ABORTADA
Motivos:
- Energia insuficiente: saldo de -4.0 kWh
Energia inicial: 80.00 kWh
Perdas: 4.00 kWh
Energia útil: 76.00 kWh
Saldo após decolagem: -4.00 kWh
Autonomia temporal não calculada.


## Entrada inválida

Este cenário omite a pressão do tanque. A validação interrompe a execução antes do cálculo energético e explica o campo ausente.

In [5]:
entrada_invalida = executar_cenario(RAIZ / 'dados' / 'entrada_invalida.json', LIMITES_PADRAO)
print(formatar_resultado(entrada_invalida))

Cenário: entrada_invalida.json
Decisão: DECOLAGEM ABORTADA
Motivos:
- Entrada inválida: campo obrigatório ausente: pressao_tanque_kpa
Energia: indisponível.


## Integração concluída

O notebook usa os módulos de missão, validação, energia, verificação e apresentação. A geração por IA é opcional: os JSONs fixos mantêm esta demonstração reproduzível sem credenciais nem internet.

## Gerador opcional: dados aleatórios do modelo
Sem seed fixa, cada execução faz novos sorteios. Os valores são preservados: dados fora das faixas podem abortar a missão.

In [6]:
from src.geracao import CENARIOS, gerar_cenario
from src.missao import executar_cenario, LIMITES_PADRAO
from importlib import reload
import src.apresentacao as apresentacao

# Atualiza o módulo mesmo quando o kernel já estava aberto antes da edição.
reload(apresentacao)

for cenario in CENARIOS:
    dados = gerar_cenario(cenario)
    resultado = executar_cenario(dados, LIMITES_PADRAO)
    apresentacao.exibir_cenario_gerado(cenario, dados, resultado)



=== Nominal ===
Temperatura: interna 22.24 °C | externa -107.75 °C
Integridade: 1 | Energia: 85.8% | Pressão: 96.17 kPa
Módulos: suporte vida=OK, energia=OK, comunicacao=OK, propulsao=OK, navegacao=OK
Decisão: PRONTO PARA DECOLAR
Nenhuma falha operacional identificada.
Energia inicial: 85.80 kWh
Perdas: 4.29 kWh
Energia útil: 81.51 kWh
Saldo após decolagem: 61.51 kWh
Autonomia estimada: 6.15 h

=== Energia insuficiente ===
Temperatura: interna 21.18 °C | externa -110.18 °C
Integridade: 0 | Energia: 45.0% | Pressão: 99.52 kPa
Módulos: suporte vida=OK, energia=OK, comunicacao=OK, propulsao=OK, navegacao=OK
Decisão: DECOLAGEM ABORTADA
Motivos:
- energia_pct 45.0 abaixo do minimo de 50
- Integridade estrutural 0, esperado NOMINAL ou 1
- Energia insuficiente: saldo de -57.25 kWh
Energia inicial: 45.00 kWh
Perdas: 2.25 kWh
Energia útil: 42.75 kWh
Saldo após decolagem: -57.25 kWh
Autonomia temporal não calculada.

=== Falha modulo ===
Temperatura: interna 20.07 °C | externa -111.78 °C
Integr